In [3]:
import boto3

# # Assume into OrganizationAccountAccessRole in a target account
# # By default this uses your *current* account ID; change TARGET_ACCOUNT_ID
# # if you want to assume into a different AWS account.
base_session = boto3.session.Session()
sts = base_session.client("sts")

current_identity = sts.get_caller_identity()
print(current_identity)



print("AWS STS get_caller_identity() after assume-role:")
print(f"  Account:   {current_identity['Account']}")
print(f"  UserId:    {current_identity['UserId']}")
print(f"  ARN:       {current_identity['Arn']}")
print(f"  Region:    {base_session.region_name}")


{'UserId': 'AIDAX4GAO7NCBZDMKVFDS', 'Account': '541569514308', 'Arn': 'arn:aws:iam::541569514308:user/christiaan.vanderberg', 'ResponseMetadata': {'RequestId': 'cc3d3c35-af5d-4aaa-ad53-548e6df4709e', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'cc3d3c35-af5d-4aaa-ad53-548e6df4709e', 'x-amz-sts-extended-request-id': 'MTpldS13ZXN0LTE6UzoxNzY4MjQ5MDk4MTAzOlI6MGFieUZ1c0o=', 'content-type': 'text/xml', 'content-length': '418', 'date': 'Mon, 12 Jan 2026 20:18:18 GMT'}, 'RetryAttempts': 0}}
AWS STS get_caller_identity() after assume-role:
  Account:   541569514308
  UserId:    AIDAX4GAO7NCBZDMKVFDS
  ARN:       arn:aws:iam::541569514308:user/christiaan.vanderberg
  Region:    eu-west-1


In [6]:
import boto3
import gzip

BUCKET = "comotion-comodash-logs"
KEY = "alb-prod/AWSLogs/541569514308/elasticloadbalancing/eu-west-1/2025/12/06/541569514308_elasticloadbalancing_eu-west-1_app.ComoDash-Prod.c90cdae921782574_20251206T0000Z_52.31.245.138_8yxysaqi.log.gz"

# Use your normal credentials (account 541569514308)
base_session = boto3.session.Session(region_name="eu-west-1")
s3 = base_session.client("s3")

obj = s3.get_object(Bucket=BUCKET, Key=KEY)
compressed_body = obj["Body"].read()

log_text = gzip.decompress(compressed_body).decode("utf-8", errors="replace")
for i, line in enumerate(log_text.splitlines(), start=1):
    print(line)
    if i >= 100:
        print("... (truncated, only first 100 lines shown)")
        break

https 2025-12-05T23:55:58.513852Z app/ComoDash-Prod/c90cdae921782574 64.252.85.107:5800 - -1 -1 -1 302 - 5620 910 "GET https://rgasa.comodash.io:443/comodash/login_check HTTP/1.1" "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36" ECDHE-RSA-AES128-GCM-SHA256 TLSv1.2 arn:aws:elasticloadbalancing:eu-west-1:541569514308:targetgroup/Comodash-rgasa/ac1fd0cda25fabd1 "Root=1-6933710e-509bd289374fc848611f2c36" "rgasa.comodash.io" "session-reused" 248 2025-12-05T23:55:58.494000Z "waf,authenticate" "-" "-" "-" "-" "-" "-" TID_685763921d671c488491483a17bc2947 "-" "-" "-"
https 2025-12-05T23:56:58.532153Z app/ComoDash-Prod/c90cdae921782574 64.252.85.107:5800 - -1 -1 -1 302 - 5532 910 "GET https://rgasa.comodash.io:443/comodash/refresh_tokens HTTP/1.1" "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36" ECDHE-RSA-AES128-GCM-SHA256 TLSv1.2 arn:aws:elasticloadbalancing:eu-west-

In [ ]:
import boto3
import gzip
from datetime import datetime, timezone

BUCKET = "comotion-comodash-logs"
BASE_PREFIX = "alb-prod/AWSLogs/541569514308/elasticloadbalancing/eu-west-1/2025/12/"

start_day = 5
end_day = 6

start_dt = datetime(2025, 12, start_day, 0, 0, 0, tzinfo=timezone.utc)
end_dt = datetime(2025, 12, end_day, 23, 59, 59, tzinfo=timezone.utc)

output_path = f"alb_logs_2025-12-{start_day}_to_2025-12-{end_day}.log"

# Use your 541569514308 user
base_session = boto3.session.Session(region_name="eu-west-1")
s3 = base_session.client("s3")

all_lines = []
MAX_LINES = 100

for day in range(start_day, end_day + 1):
    if len(all_lines) >= MAX_LINES:
        break
    day_prefix = f"{BASE_PREFIX}{day:02d}/"  # <-- pad with leading zero
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET, Prefix=day_prefix):
        if len(all_lines) >= MAX_LINES:
            break
        for obj in page.get("Contents", []):
            if len(all_lines) >= MAX_LINES:
                break
            key = obj["Key"]

            body = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
            text = gzip.decompress(body).decode("utf-8", errors="replace")

            for line in text.splitlines():
                if len(all_lines) >= MAX_LINES:
                    break
                if not line.strip():
                    continue
                if "session-reused" in line:
                    continue
                if "https://rgasa.comodash.io" not in line and "comoauth" not in line:
                    continue
                # ALB timestamp is second field, e.g. 2025-12-05T23:55:58.513852Z
                fields = line.split()
                if len(fields) < 2:
                    continue
                ts_str = fields[1]
                try:
                    ts = datetime.fromisoformat(ts_str.replace("Z", "+00:00"))
                except ValueError:
                    continue
                if start_dt <= ts <= end_dt:
                    all_lines.append(line)

with open(output_path, "w", encoding="utf-8") as f:
    for line in all_lines[:MAX_LINES]:
        f.write(line + "\n")

print(
    f"Wrote {len(all_lines)} ALB log lines (capped at {MAX_LINES}) from s3://{BUCKET}/{BASE_PREFIX}[{start_day}-{end_day}] "
    f"between {start_dt} and {end_dt} to {output_path}."
)